In [2]:
!pip install gym==0.23.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.2/626.2 kB 5.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gym: filename=gym-0.23.1-py3-none-any.whl size=701459 sha256=487c1f0356ca89b1544dae5d150309d96ccf3e7aa0f3e9e859d54c49ae5d29bc
  Stored in directory: /root/.cache/pip/wheels/76/3d/86/444fdf01f8cf84453dd21d0aa0c4c4835a2cf7183b810e5a53
Successfully built gym
  Attempting uninstall: gym
    Found existing installation: gym 0.25.2
    Uninstalling gym-0.25.2:
      Successfully uninstalled gym-0.25.2


In [3]:
import gym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
entorno = gym.make('Taxi-v3')

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

In [21]:
S = entorno.reset()
entorno.render()
print('Estado inicial: {}'.format(S))
print('Numero de estados: {}'.format(entorno.observation_space.n))
print('Numero de acciones: {}'.format(entorno.action_space.n))

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+

Estado inicial: 62
Numero de estados: 500
Numero de acciones: 6


In [15]:
import numpy as np
import matplotlib.pyplot as plt

In [19]:
def mostrar_estado(env,S):
  #decodeificar el estado

  tx_fil, tx_col, origen, destino = entorno.env.decode(S)
  origenes = {0: 'R(ed)', 1: 'G(reen)', 2: 'Y(ellow)', 3: 'B(lue)', 4: 'In Taxi'}
  destinos = {0: 'R(ed)', 1: 'G(reen)', 2: 'Y(ellow)', 3: 'B(lue)'}

  print(f'Ubicación del taxi (Fil, col): {tx_fil+1, tx_col+1}')
  print(f'Origen: {origenes[origen]}')
  print(f'Destino: {destinos[destino]}')

In [22]:
mostrar_estado(entorno,S)

Ubicación del taxi (Fil, col): (1, 4)
Origen: R(ed)
Destino: Y(ellow)


In [59]:
acciones = {0:'abajo', 1:'arriba', 2:'izquierda', 3:'derecha', 4:'recoger', 5:'dejar'}

A = entorno.action_space.sample()
S, R, done, info = entorno.step(A)
entorno.render()
print(A)
print(S)
print(R)
print(done)
mostrar_estado(entorno,S)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)
1
42
-1
False
Ubicación del taxi (Fil, col): (1, 3)
Origen: R(ed)
Destino: Y(ellow)


In [80]:
#Funcion inicializar Q nS: num estados, nA: num acciones
def inicializar_Q(nS, nA, tipo = 'ones'):
  if tipo == 'ones':
    Q = np.ones((nS, nA))
  elif tipo == 'random':
    Q = np.random.randint((nS, nA))
  return Q

In [81]:
Q = inicializar_Q(10,5, tipo = 'ones')
print(Q.shape)
print(Q)

(10, 5)
[[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]


In [82]:
#permite seleccionar una accion de la tabla Q usando el enfoque e-greedy
def e_greedy(s, Q, e, nA):
  if np.random.random() >= e: #prob 1- epsilon
    a = np.argmax(Q[s,:])
  else: #prob de epsiolon
    a = np.random.randint(0, nA)
  return a

In [83]:
s = 3
e = 0.2
a = e_greedy(s, Q, e, 3)
print(f'tabla Q para s = {s}: {Q[s,:]}')
print(f'accion seleccionada para e={e}: es {a}')

tabla Q para s = 3: [1. 1. 1. 1. 1.]
accion seleccionada para e=0.2: es 0


In [88]:
def Sarsa_onpolicy(entorno, alpha, gamma, epsilon, nS, nA, k, verbose=True):
  Q = inicializar_Q(nS, nA)
  retorno = []

  for episodio in range(k):
    retorno_acumulado = 0
    s = entorno.reset()
    a = e_greedy(s, Q, epsilon, nA)
    done = False #no estado terminal

    while not done:
      s_, r_, done, info = entorno.step(a)
      a_ = e_greedy(s, Q, epsilon, nA)
      retorno_acumulado += r_

      if not done: #si no es estado terminal
        Q[s, a] += alpha * (r_ + gamma * Q[s_, a_] - Q[s, a])
      else:
        Q[s, a] += alpha * (r_ - Q[s, a])
        retorno.append(retorno_acumulado)
      s, a = s_, a_
      if verbose:
        if episodio%100 == 0:
          print(f'episodio {episodio+1}/{k}:')
          print(f'\t\t Recompensa {r_}')

  return Q, retorno

In [89]:
ALPHA = 0.4
GAMMA = 0.999
EPSILON = 0.1
K = 1000
NS= entorno.observation_space.n
NA = entorno.action_space.n
Q_s, ret_s = Sarsa_onpolicy(entorno, ALPHA, GAMMA, EPSILON, NS, NA, K)

episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -10
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -10
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -10
episodio 1/1000:
		 Recompensa -10
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1/1000:
		 Recompensa -1
episodio 1

In [90]:
#sarsa y q learning para la tarea que se esta haciendo